In [2]:
import json
import sys
sys.path.append(r"C:\Users\Abhinav Arora\Downloads")
with open(r"C:\Users\Abhinav Arora\Downloads\coco_wholebody_train_v1.0.json", "r") as file:
    data = json.load(file)

In [3]:
#Reconfiguring the JSON file to remove unecessary data from the annotations:
keys_to_keep = set(["id", "image_id", "category_id", "segmentation", "area", "bbox", "iscrowd", "keypoints", "num_keypoints", "foot_valid"])

#Indices for left big toe in foot_kpts = 0, 1, 2
#Indices for right big toe in foot_kpts = 9, 10, 11

annotations = []
for a in data["annotations"]:
    copy_a = a.copy()
    # Zeroing out the foot coordinates in case they are garbage values and foot_valid is false
    if copy_a["foot_valid"] == False:
        copy_a["foot_kpts"] = [0] * 18
    
    left_big_toe_kpts = copy_a["foot_kpts"][0:3]
    right_big_toe_kpts = copy_a["foot_kpts"][9:12]
    copy_a["keypoints"] = copy_a["keypoints"] + left_big_toe_kpts + right_big_toe_kpts

    keys_to_remove = [key for key in copy_a if key not in keys_to_keep]
    for key in keys_to_remove:
        del copy_a[key]
    
    keypoint_count = 0
    #Going through all keypoints to evaluate their validity. It is always the third index
    for i in range(1, len(a["keypoints"])//3):
        ind = (i * 3) - 1
        if copy_a["keypoints"][ind] != 0:
            keypoint_count += 1
    
    copy_a["num_keypoints"] = keypoint_count
    annotations.append(copy_a)

In [4]:
#Reassigning annotations:
data["annotations"] = annotations

keys in annotation:

segmentation
num_keypoints
area
iscrowd
keypoints
image_id
bbox
category_id
id
face_box
lefthand_box
righthand_box
lefthand_kpts
righthand_kpts
face_kpts
face_valid
lefthand_valid
righthand_valid
foot_valid
foot_kpts

following shows the revised keypoints:
{
    "id": int,                    # unique annotation id
    "image_id": int,              # links to an entry in "images"
    "category_id": int,           # 1 for person
    "segmentation": [...],        # polygon segmentation (can ignore)
    "area": float,                # bounding box area
    "bbox": [x, y, w, h],        # bounding box
    "iscrowd": 0,                 # 0 for individual, 1 for crowd
    "keypoints": [x1,y1,v1, x2,y2,v2, ... x17,y17,v17],  # 51 values flat + 6 values for each left and right big toe
    "num_keypoints": int          # number of labeled keypoints -> 19 total for the 2 toes as well (maximum possible)
}